# Council Report Generator (SAcommunity + Google Analytics)

This script combines data from three source files to produce a clean Power BI Excel file containing organisation-level engagement metrics for a given council and financial year.

**What you need to do:**
* Place this script and all input Excel files in the same folder.
* Update the council name, year, and filenames in the **Extract Data** cell below.
* Run all cells top to bottom. The output file will be saved in the same folder.

**Notes for non-technical users:**
* `Organisation ID` is a unique number for each organisation on SAcommunity.
* Google Analytics stores pages like `/org/12345-sample_org`. We extract:
    - `12345` -> Organisation ID
    - `sample org` -> Organisation Name (underscores converted to spaces)
* Organisations in SAcommunity with no GA activity are included with zero values so reports are complete.

For any issues contact Anubhav: ada@sacommunity.org

In [1]:
import pandas as pd
import numpy as np
import re

## Extract Data

Update the four variables below before running the script.

1. Export the Excel files from SAcommunity.org and Google Looker Studio.
2. Place all files in the same folder as this script.
3. Set `Council_Name`, `year`, and the three filenames to match your exports.

In [2]:
# Council name -- e.g. "Holdfast Bay"
Council_Name = "Holdfast Bay"

# Financial year label -- e.g. "25_26"
year = "25_26"

# CiviCRM export for the council from SAcommunity
# e.g. "Holdfast Bay_25_26_CiviCRM.xlsx"
CiviCRM_Filename = "Holdfast_25_26_CiviCRM.xlsx"

# Data.gov / cu_export for the council from SAcommunity
# e.g. "Holdfast Bay_25_26_DataGov_export.xlsx"
Data_gov_Filename = "Holdfast_25_26_Data.Gov.au_export.xlsx"

# Full-year Google Analytics export
# e.g. "SAcommunity_GA_25_26_FinancialYr_Export.xlsx"
GA_Filename = "25_26_GA4_All year.xlsx"

## Load & Standardise Source Tables

We read each Excel file into a pandas DataFrame and rename columns to a consistent standard (Australian spelling throughout).

Each source uses slightly different column names:
- **Data.gov**: `ID_19` -> `Organisation ID`, `Org_name` -> `Organisation Name`
- **CiviCRM**: `Internal Contact ID` -> `Organisation ID`, `Organization Name` -> `Organisation Name`
- **GA**: `Organisation ` (trailing space) -> `Organisation Name`, `Landing page + query string` -> `Landing page`

We also parse GA landing pages (e.g. `/org/12345-sample_org`) to extract `Organisation ID` and `Organisation Name`.

In [ ]:
# --- Data.gov export ---
Data_gov_df = pd.read_excel(Data_gov_Filename)
Data_gov_df.rename(
    columns={
        'ID_19': 'Organisation ID',
        'Org_name ': 'Organisation Name'   # note: trailing space in source column
    },
    inplace=True
)

# --- CiviCRM export (pre-filtered to the target council) ---
CiviCRM_df = pd.read_excel(CiviCRM_Filename)
CiviCRM_df.rename(
    columns={
        'Internal Contact ID': 'Organisation ID',
        'Organization Name': 'Organisation Name'  # American -> Australian spelling
    },
    inplace=True
)
CiviCRM_df = CiviCRM_df[['Organisation Name', 'Organisation ID', 'Primary Category']]

# --- Google Analytics export ---
GA_df = pd.read_excel(GA_Filename)
GA_df.rename(
    columns={
        'Organisation ': 'Organisation Name',      # trailing space in source column
        'Landing page + query string': 'Landing page'
    },
    inplace=True
)

# Keep only rows that represent an organisation page (e.g. /org/12345-sample_org)
GA_df['Landing page'] = GA_df['Landing page'].astype(str)
GA_df = GA_df[GA_df['Landing page'].str.contains(r'/org/\d+', regex=True)]

# Extract Organisation ID from the URL path
#GA_df['Organisation ID'] = (
    #GA_df['Landing page'].str.extract(r'/org/(\d+)-')[0].astype('int64')) # This code works with previous years (Before FY 2025-2026)
# NB: Since there are two type of Organisation ID formats in the FY 2025-2026 all year data (i.e., /org/216954-Brougham_Place_Uniting_Church; /org/216954),
# we need to remove NaNs generated from IDs that do not have the name of the council after their IDs (i.e., /org/216954).
# Otherwise, errors will occur from astype('int64'). Edited by Lam Nguyen.

GA_df['Organisation ID'] = GA_df['Landing page'].str.extract(r'/org/(\d+)-')[0]
GA_df = GA_df.dropna(subset=['Organisation ID'])
GA_df['Organisation ID'] = GA_df['Organisation ID'].astype('int64')



def parse_org_name_from_url(url_series: pd.Series) -> pd.Series:
    """
    Convert GA landing page paths into readable organisation names.

    Example:
        '/org/12345-sample_organisation'  ->  'sample organisation'

    Steps:
        1. Strip the leading '/org/<number>-' prefix.
        2. Replace underscores with spaces.
    """
    return url_series.str.replace(r'/org/\d+-', '', regex=True).str.replace('_', ' ')


GA_df['Organisation Name'] = parse_org_name_from_url(GA_df['Landing page'])

## Build the Council Organisation List

We merge CiviCRM and Data.gov on `Organisation ID` to produce a single master list of all organisations in SAcommunity for this council.

GA data is then filtered to only those organisations, and the raw URL column is dropped.

In [4]:
# Outer merge to capture organisations present in either source
SA_Community_df = (
    pd.merge(CiviCRM_df, Data_gov_df, on='Organisation ID', how='outer')
    .drop_duplicates(subset='Organisation ID')
    [['Organisation Name', 'Organisation ID']]
)

# Filter GA to only organisations belonging to this council, then drop the raw URL
session_df = (
    GA_df[GA_df['Organisation ID'].isin(SA_Community_df['Organisation ID'])]
    .reset_index(drop=True)
    .drop(columns=['Landing page'])
)

## Add Organisations with No Google Analytics Activity

Organisations that appear in SAcommunity but have no GA traffic are added with zero values.

This ensures the final export is complete and totals are not understated.

In [5]:
# Organisations in SAcommunity that have no GA sessions
Zero_sessions_df = (
    SA_Community_df[~SA_Community_df['Organisation ID'].isin(session_df['Organisation ID'])]
    .copy()
    .reset_index(drop=True)
)

# Default values for all GA metric and dimension columns
zero_defaults = {
    'Sessions': 0,
    'New users': 0,
    'Active users': 0,
    'Total users': 0,
    'Views': 0,
    'Age': 0,
    'Gender': 'unknown',
    'Device': '(not set)',
    'Platform/Device category': 'web / desktop'
}
for col, default in zero_defaults.items():
    Zero_sessions_df[col] = default

# Column order to align with session_df before concatenating
zero_cols = ['Organisation ID', 'Organisation Name'] + list(zero_defaults.keys())

# Combine actual GA sessions with zero-activity rows
final_df = pd.concat(
    [session_df, Zero_sessions_df[zero_cols]],
    ignore_index=True
)

# Summarise by organisation: sum metrics, keep first name, sort by sessions descending
total_sessions_df = (
    final_df.groupby('Organisation ID').agg(
        {
            'Organisation Name': 'first',
            'Sessions': 'sum',
            'New users': 'sum',
            'Active users': 'sum',
            'Total users': 'sum',
            'Views': 'sum'
        }
    )
    .reset_index()
    .sort_values(by='Sessions', ascending=False)
    .reset_index(drop=True)
)

## Merge for Power BI Export

We attach the primary service category to each organisation using a two-step approach:
1. Map from CiviCRM using a numeric ID -> label dictionary.
2. Fill any remaining blanks from the Data.gov export.

The result is a single Excel file ready to load into Power BI.

In [6]:
# Numeric ID -> human-readable category label
PRIMARY_CATEGORY_MAP = {
    13115: "Accommodation",
    13186: "Animals, Birds",
    13212: "Citizenship, Nationality",
    13269: "Communication & Information Services",
    13366: "Community Organisation & Development",
    13474: "Education",
    13549: "Employment",
    13589: "Environment & Heritage",
    13659: "Finance, Income, Business",
    13741: "Government",
    13753: "Health & Disability",
    14025: "Law & Justice",
    14139: "Recreation",
    14448: "Material & Practical Needs",
    14503: "Personal & Family Support",
    14588: "Public Safety",
    14616: "Religions & Philosophies",
    14673: "Rural Organisation & Development",
    14698: "Transport"
}

# Step 1: merge primary category from CiviCRM
CiviCRM_categories = (
    CiviCRM_df[['Organisation ID', 'Primary Category']]
    .rename(columns={'Primary Category': 'Primary Category ID'})
)
merged_df = pd.merge(total_sessions_df, CiviCRM_categories, on='Organisation ID', how='left')
merged_df['Primary Category'] = merged_df['Primary Category ID'].replace(PRIMARY_CATEGORY_MAP)
merged_df = merged_df.drop(columns=['Primary Category ID'])

# Step 2: fill blanks from Data.gov using a fast index-based lookup
missing_mask = merged_df['Primary Category'].isna()
category_fallback = Data_gov_df.set_index('Organisation ID')['Primary_Category']
merged_df.loc[missing_mask, 'Primary Category'] = (
    merged_df.loc[missing_mask, 'Organisation ID'].map(category_fallback)
)

powerbi_df = merged_df.copy()

## Fill Missing Names, Clean & Export

After all merges, some organisations may still have a missing name. We fall back to the Data.gov name where available.

We then run a final name-cleaning pass to strip non-ASCII characters and restore common encoding artefacts.

Any organisation whose name still required changes after the automated fixes is flagged in `triggered_org_id` for manual review.

The file is saved as `_PowerBI_<Council_Name>_<year>_data.xlsx`.

In [7]:
def make_cleaner(trigger_list: list):
    """
    Return a row-level cleaning function that:
      1. Strips non-ASCII characters from the organisation name.
      2. Restores common encoding artefacts:
           '26'  -> '&'  (HTML entity remnant from URL encoding)
           garbled em-dash sequence  -> '-'
           garbled apostrophe sequence -> single quote
      3. If the name changed in step 1 but was NOT resolved by step 2,
         appends the Organisation ID to trigger_list for manual review.

    Returns a tuple: (cleaned_name, trigger_id_or_empty_string)
    """
    def clean_name(name: str, org_id) -> tuple:
        # Step 1: strip non-ASCII characters
        cleaned = re.sub(r"[^A-Za-z0-9 \-.,'()]", "", str(name))
        needs_review = cleaned != str(name)

        # Step 2: restore known encoding artefacts
        replacements = [
            ('26',       '&'),
            ('\u00e2\u20ac\u201c', '-'),
            ('\u00e2\u20ac\u2122', "'"),
        ]
        for artefact, correct in replacements:
            if artefact in cleaned:
                cleaned = cleaned.replace(artefact, correct)
                needs_review = False  # recognised and fixed; no manual review needed

        if needs_review:
            trigger_list.append(org_id)
            return cleaned, org_id
        return cleaned, ''

    return clean_name


# Fill missing organisation names from Data.gov
id_to_name_map = Data_gov_df.set_index('Organisation ID')['Org_name'].to_dict()
missing_mask = powerbi_df['Organisation Name'].isna()
powerbi_df.loc[missing_mask, 'Organisation Name'] = (
    powerbi_df.loc[missing_mask, 'Organisation ID'].map(id_to_name_map)
)

# Run the cleaning pass
triggered_org_ids = []
cleaner = make_cleaner(triggered_org_ids)
powerbi_df[['Organisation Name', 'triggered_org_id']] = powerbi_df.apply(
    lambda row: pd.Series(cleaner(row['Organisation Name'], row['Organisation ID'])),
    axis=1
)

if triggered_org_ids:
    print(f"Warning: {len(triggered_org_ids)} organisation(s) may need manual name review:")
    print(triggered_org_ids)

# Export
output_filename = f"_PowerBI_{Council_Name}_{year}_data.xlsx"
powerbi_df.to_excel(output_filename, index=False)
print(f"Report saved: {output_filename}")

[217601]
Report saved: _PowerBI_Holdfast Bay_25_26_data.xlsx
